In [1]:
from IPython.display import display, HTML
display(HTML('''
<style>
.jp-Cell-outputWrapper .jp-Placeholder {
    display: none;
}
</style>
'''))

<center>
    <img src="https://upload.wikimedia.org/wikipedia/commons/a/a8/%D0%9B%D0%9E%D0%93%D0%9E_%D0%A8%D0%90%D0%94.png" width=500px/>
    <font>Python 2025</font><br/>
    <br/>
    <br/>
    <b style="font-size: 2em">Типы - Часть 2</b><br/>
    <br/>
    <font>Никита Бондарцев</font><br/>
</center>

### Что мы сегодня разберем?

### Специальный тип Any

In [2]:
import typing as tp
print(tp.Any.__doc__.split("\n")[0])

Special type indicating an unconstrained type.


### Почему тип Any специальный?

In [3]:
import typing as tp

# Тип-класс
class A:
    pass

# Тип-класс
B = int

# Тип-объект
C = tp.Any

type(A), type(B), type(C)

(type, type, typing._AnyMeta)

In [4]:
isinstance(1, tp.Any)

TypeError: typing.Any cannot be used with isinstance()

### Специальный тип Union

In [5]:
import typing as tp
# нынче, с питона 3.10, можно писать просто через вертикальную черту A | B
print(tp.Union.__doc__.split("\n")[0])

Union type; Union[X, Y] means either X or Y.


Определение подтипа:
* ∀ A: A -> A
* A -> B  =>  A.values ⊇ B.values
* A -> B  =>  A.functions ⊆ B.functions

float | str => объединение всех значений, пересечение всех методов, поэтому

#### Выполняются ли такие свойства?

float | str -> float \
float | str -> str

In [6]:
%%typecheck
from typing import reveal_type

class A:
    def am(self) -> None:
        pass
    
    def run(self) -> None:
        pass

class B:
    def run(self) -> None:
        pass


a: A | B = A()
reveal_type(a)
a.am()
a.run()

 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmp9h38ovfm.py:17:12-15: revealed type: A | B [reveal-type]
ERROR /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmp9h38ovfm.py:18:1-5: Object of class `B` has no attribute `am` [missing-attribute]

 INFO 1 error



In [7]:
float | str == str | float, tp.Union[str, float] == str | float

(True, True)

In [8]:
float == tp.Union[float]

True

### Union → Примеры

In [9]:
%%typecheck
# Передача подтипов в Union
from typing import reveal_type

def f(a: float | str) -> None:
    pass

f(2)
f(2.0)
f("hello")
f({})

ERROR /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpw7j2fzr8.py:11:3-5: Argument `dict[@_, @_]` is not assignable to parameter `a` with type `float | str` in function `f` [bad-argument-type]

 INFO 1 error



In [10]:
%%typecheck
# Передача надтипов в Union

def f(a: int | str) -> None:
    pass

f(2)
f(2.0)
f("hello")
f({})

ERROR /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpp2w__59e.py:8:3-6: Argument `float` is not assignable to parameter `a` with type `int | str` in function `f` [bad-argument-type]
ERROR /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpp2w__59e.py:10:3-5: Argument `dict[@_, @_]` is not assignable to parameter `a` with type `int | str` in function `f` [bad-argument-type]

 INFO 2 errors



In [11]:
%%typecheck
# Вывод типов с учетом isinstance
from typing import reveal_type


def f(a: int | float | str) -> None:
    reveal_type(a)
    if isinstance(a, int):
        reveal_type(a)
    else:
        reveal_type(a)
        if isinstance(a, str):
            reveal_type(a)
        else:
            reveal_type(a)


 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpp9_j0a1m.py:7:16-19: revealed type: float | int | str [reveal-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpp9_j0a1m.py:9:20-23: revealed type: int [reveal-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpp9_j0a1m.py:11:20-23: revealed type: float | str [reveal-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpp9_j0a1m.py:13:24-27: revealed type: str [reveal-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpp9_j0a1m.py:15:24-27: revealed type: float [reveal-type]

 INFO 0 errors



#### Some really weird things (will be discussed again later at the end of the lecture, but it would still be not clear, sorry)

- [virtual subclassing of ABC](https://docs.python.org/3.13/library/abc.html#abc.ABCMeta.register)
- the [numbers module](https://docs.python.org/3.13/library/numbers.html) and ABC hierarchy, [pep for numeric tower](https://peps.python.org/pep-3141/)
- source code that [registers float](https://github.com/python/cpython/blob/3.13/Lib/numbers.py#L289) as a virtual subclass of numbers.Real
- isinstance() would not work as expected!

In [12]:
class A: pass
class B(A): pass


issubclass(int, float), isinstance(2, float), isinstance(2., float), issubclass(B, A), isinstance(B(), A)


(False, False, True, True, True)

In [13]:
int_method_names = set(int.__dict__.keys())
float_method_names = set(float.__dict__.keys())
int_method_names - float_method_names, float_method_names - int_method_names

({'__and__',
  '__getattribute__',
  '__index__',
  '__invert__',
  '__lshift__',
  '__or__',
  '__rand__',
  '__rlshift__',
  '__ror__',
  '__rrshift__',
  '__rshift__',
  '__rxor__',
  '__sizeof__',
  '__xor__',
  'bit_count',
  'bit_length',
  'denominator',
  'from_bytes',
  'numerator',
  'to_bytes'},
 {'__getformat__', 'fromhex', 'hex'})

In [14]:
%%typecheck
# Вывод типов с учетом isinstance
from typing import reveal_type


def f(a: int | float | str) -> None:
    reveal_type(a)
    if isinstance(a, float):
        reveal_type(a)
        if isinstance(a, str):
            reveal_type(a)
    else:
        reveal_type(a)
        if isinstance(a, str):
            reveal_type(a)
        else:
            reveal_type(a)

 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmp87o7f8qb.py:7:16-19: revealed type: float | int | str [reveal-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmp87o7f8qb.py:9:20-23: revealed type: float | int [reveal-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmp87o7f8qb.py:11:24-27: revealed type: str [reveal-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmp87o7f8qb.py:13:20-23: revealed type: str [reveal-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmp87o7f8qb.py:15:24-27: revealed type: str [reveal-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmp87o7f8qb.py:17:24-27: revealed type: Never [reveal-type]

 INFO 0 errors



### Специальный тип Optional

In [15]:
import typing as tp
print(tp.Optional.__doc__.split("\n")[0])

Optional[X] is equivalent to Union[X, None].


In [16]:
import typing as tp
from types import NoneType  # python 3.10+

tp.Optional[float] == float | None, tp.Optional[float] == float | NoneType, type(None), type(None) is NoneType

(True, True, NoneType, True)

### Специальный тип Optional, примеры

In [17]:
%%typecheck

def f(a: float | None) -> None:
    pass

f(1)
f(1.0)
f("1")
f(None)

ERROR /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmp3rgzl2le.py:8:3-6: Argument `Literal['1']` is not assignable to parameter `a` with type `float | None` in function `f` [bad-argument-type]

 INFO 1 error



In [18]:
%%typecheck
# reveal в Optional делается аналогично Union
from typing import reveal_type


def f(a: float | None) -> None:
    reveal_type(a)
    if a is None:
        reveal_type(a)
    else:
        reveal_type(a)

 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpf4eq5ktw.py:7:16-19: revealed type: float | None [reveal-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpf4eq5ktw.py:9:20-23: revealed type: None [reveal-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpf4eq5ktw.py:11:20-23: revealed type: float [reveal-type]

 INFO 0 errors



### Что такое генерики?

Генерик - параметризированный тип, [ссылка на доку](https://mypy.readthedocs.io/en/stable/generics.html#defining-generic-classes)

Например, \
List[int] - тип list с параметром int \
Dict[str, int] - dict c параметром str у ключа и int у значения \
Tuple[int, ...] - tuple с параметрами int

Note: с питона 3.9 можно использовать стандартные контейнеры в качестве генериков, например, list[int], dict[str, int], tuple[int, float, str]

### Ковариантность/контрвариантность

### Примеры вариантности

### tuple (typing.Tuple, deprecated)

In [19]:
from typing import Tuple
print(Tuple.__doc__)

Deprecated alias to builtins.tuple.

    Tuple[X, Y] is the cross-product type of X and Y.

    Example: Tuple[T1, T2] is a tuple of two elements corresponding
    to type variables T1 and T2.  Tuple[int, float, str] is a tuple
    of an int, a float and a string.

    To specify a variable-length tuple of homogeneous type, use Tuple[T, ...].
    


In [20]:
%%typecheck
import typing as tp

a: tp.Tuple[int, str] = (1, "hello")
b: tuple[int, int, int] = (1, 2, 3)
c: tuple[int, ...] = (1, 2, 3, 4, 5)
d: tuple[int, float, str] = (1, 2.)

ERROR /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpdbxw02hr.py:7:29-36: `tuple[Literal[1], float]` is not assignable to `tuple[int, float, str]` [bad-assignment]

 INFO 1 error



### tuple, пример

In [21]:
%%typecheck
# Heterogeneus tuple 

def f(a: tuple[int, float]) -> None:
    a[0] << 10
    a[1] << 10
    a[2] + 1

f((1, 1.4))
f((1, 1.4, 1))
f((1, 1))
f((1.4, 1))
f((1, "1"))

ERROR /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpqyxc9zr0.py:6:5-15: `<<` is not supported between `float` and `Literal[10]` [unsupported-operation]
ERROR /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpqyxc9zr0.py:7:5-9: Index 2 out of range for tuple with 2 elements [index-error]
ERROR /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpqyxc9zr0.py:10:3-14: Argument `tuple[Literal[1], float, Literal[1]]` is not assignable to parameter `a` with type `tuple[int, float]` in function `f` [bad-argument-type]
ERROR /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpqyxc9zr0.py:12:3-11: Argument `tuple[float, Literal[1]]` is not assignable to parameter `a` with type `tuple[int, float]` in function `f` [bad-argument-type]
ERROR /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpqyxc9zr0.py:13:3-11: Argument `tuple[Literal[1], Literal['1']]` is not assignable to parameter `a` with type `tuple[int, float]` in function `f` [bad-argument-type]

 INFO 5 errors



In [22]:
%%typecheck
# Homogeneus tuple 

def f(a: tuple[float, ...]) -> None:
    a[0] + 1
    a[1] << 10
    a[100500] = 1.54

f((1, 2, 3, 4, 5))
f((1.1, 2.1, 3.1, 4.1, 5.1))
f(("3", "2", "1"))
f(tuple())

ERROR /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmp9cqt63xi.py:6:5-15: `<<` is not supported between `float` and `Literal[10]` [unsupported-operation]
ERROR /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmp9cqt63xi.py:7:5-14: Cannot set item in `tuple[float, ...]` [unsupported-operation]
ERROR /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmp9cqt63xi.py:11:3-18: Argument `tuple[Literal['3'], Literal['2'], Literal['1']]` is not assignable to parameter `a` with type `tuple[float, ...]` in function `f` [bad-argument-type]

 INFO 3 errors



### Какая вариантность у tuple[T]?

### list (typing.List, not deprecated, hmm)

In [23]:
import typing as tp
print(tp.List.__doc__)

A generic version of list.


### list, примеры

In [24]:
%%typecheck
from typing import reveal_type

a = [1, "hello", 5.1]
reveal_type(a)

a.append(1)
a.append(1.0)
a.append("hi")
a.append([])

 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpxo6t5uc2.py:5:12-15: revealed type: list[float | int | str] [reveal-type]
ERROR /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpxo6t5uc2.py:10:10-12: Argument `list[@_]` is not assignable to parameter `object` with type `float | int | str` in function `list.append` [bad-argument-type]

 INFO 1 error



In [25]:
%%typecheck
from typing import reveal_type

a = []
reveal_type(a)

 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpkg9u1xkv.py:5:12-15: revealed type: list[@_] [reveal-type]

 INFO 0 errors



In [26]:
%%typecheck
from typing import reveal_type
# meeeh, don't like pyrefly =/
a = []
reveal_type(a)
a.append(3.1415)
reveal_type(a)
a.append("hi")
reveal_type(a)


 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpo6dzag1c.py:5:12-15: revealed type: list[@_] [reveal-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpo6dzag1c.py:7:12-15: revealed type: list[Unknown] [reveal-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpo6dzag1c.py:9:12-15: revealed type: list[Unknown] [reveal-type]

 INFO 0 errors



In [27]:
%%typecheck
from typing import reveal_type

a: list[float] = []

a.append(1)
reveal_type(a)
b = a[0]
reveal_type(b)

 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmprmx82tii.py:7:12-15: revealed type: list[float] [reveal-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmprmx82tii.py:9:12-15: revealed type: float [reveal-type]

 INFO 0 errors



In [28]:
%%typecheck
from typing import reveal_type

a: list[int] = []

a.append(1.1)
reveal_type(a)

ERROR /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpraaiyqu6.py:6:10-13: Argument `float` is not assignable to parameter `object` with type `int` in function `list.append` [bad-argument-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpraaiyqu6.py:7:12-15: revealed type: list[int] [reveal-type]

 INFO 1 error



In [29]:
%%typecheck
# Повышение типа параметра
from typing import reveal_type

def foo(a: list[int]) -> None:
    pass

my_list = [1.1, 3.1, 5.1]
reveal_type(my_list)

foo(my_list)

 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpnhq5j5_b.py:9:12-21: revealed type: list[float] [reveal-type]
ERROR /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpnhq5j5_b.py:11:5-12: Argument `list[float]` is not assignable to parameter `a` with type `list[int]` in function `foo` [bad-argument-type]

 INFO 1 error



In [30]:
%%typecheck
# Понижение типа параметра
from typing import reveal_type

def foo(a: list[float]) -> None:
    pass

my_list = [1, 2, 3]
reveal_type(my_list)

foo(my_list)

 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmp7v92nb2g.py:9:12-21: revealed type: list[int] [reveal-type]
ERROR /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmp7v92nb2g.py:11:5-12: Argument `list[int]` is not assignable to parameter `a` with type `list[float]` in function `foo` [bad-argument-type]

 INFO 1 error



### Какая вариантность у list[T]?

### typing.Sequence/typing.Mapping

### Sequence, пример

In [31]:
%%typecheck

def foo(a: list[float]) -> float:
    return a[0]

my_list = [1, 3, 5]

foo(my_list)

ERROR /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpd9ely0jf.py:8:5-12: Argument `list[int]` is not assignable to parameter `a` with type `list[float]` in function `foo` [bad-argument-type]

 INFO 1 error



In [32]:
%%typecheck

import collections.abc as abc

def foo(a: abc.Sequence[float]) -> float:
    return a[0]

my_list = [1, 3, 5]

foo(my_list)

 INFO 0 errors



### abc.Mapping, пример

In [33]:
%%typecheck

def foo(a: dict[str, float]) -> float:
    return a["key"]

my_dict = {"hey": 1}

foo(my_dict)

ERROR /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpojw1sh9g.py:8:5-12: Argument `dict[str, int]` is not assignable to parameter `a` with type `dict[str, float]` in function `foo` [bad-argument-type]

 INFO 1 error



In [34]:
%%typecheck

import collections.abc as abc

def foo(a: abc.Mapping[str, float]) -> float:
    return a["key"]

my_dict = {"hey": 1}

foo(my_dict)

 INFO 0 errors



### Иерархия типов генериков
<img src="images/hierarchy2.jpg">

### Примеры на иерархию abc.Sequence/abc.Mapping 

In [35]:
%%typecheck

import collections.abc as abc


def foo1(a: abc.Sequence[float]) -> None:
    pass

def foo2(a: abc.MutableSequence[float]) -> None:
    pass

def foo3(a: list[float]) -> None:
    pass

def foo4(a: list) -> None:
    pass


a = [1]
foo1(a)
foo2(a)
foo3(a)
foo4(a)


ERROR /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmp9qtbfurt.py:21:6-7: Argument `list[int]` is not assignable to parameter `a` with type `MutableSequence[float]` in function `foo2` [bad-argument-type]
ERROR /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmp9qtbfurt.py:22:6-7: Argument `list[int]` is not assignable to parameter `a` with type `list[float]` in function `foo3` [bad-argument-type]

 INFO 2 errors



In [36]:
%%typecheck

import collections.abc as abc


def foo1(a: abc.Mapping[str, float]) -> None:
    pass

def foo2(a: abc.MutableMapping[str, float]) -> None:
    pass

def foo3(a: dict[str, float]) -> None:
    pass

def foo4(a: dict) -> None:
    pass


a = {'a': 1}
foo1(a)
foo2(a)
foo3(a)
foo4(a)

ERROR /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmp5dxxpujg.py:21:6-7: Argument `dict[str, int]` is not assignable to parameter `a` with type `MutableMapping[str, float]` in function `foo2` [bad-argument-type]
ERROR /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmp5dxxpujg.py:22:6-7: Argument `dict[str, int]` is not assignable to parameter `a` with type `dict[str, float]` in function `foo3` [bad-argument-type]

 INFO 2 errors



### Generic functions, TypeVar

In [37]:
print(tp.TypeVar.__doc__)

Type variable.

The preferred way to construct a type variable is via the dedicated
syntax for generic functions, classes, and type aliases::

    class Sequence[T]:  # T is a TypeVar
        ...

This syntax can also be used to create bound and constrained type
variables::

    # S is a TypeVar bound to str
    class StrSequence[S: str]:
        ...

    # A is a TypeVar constrained to str or bytes
    class StrOrBytesSequence[A: (str, bytes)]:
        ...

Type variables can also have defaults:

    class IntDefault[T = int]:
        ...

However, if desired, reusable type variables can also be constructed
manually, like so::

   T = TypeVar('T')  # Can be anything
   S = TypeVar('S', bound=str)  # Can be any subtype of str
   A = TypeVar('A', str, bytes)  # Must be exactly str or bytes
   D = TypeVar('D', default=int)  # Defaults to int

Type variables exist primarily for the benefit of static type
checkers.  They serve as the parameters for generic types as well
as for generic func

#### TypeVar, мотивация

In [38]:
%%typecheck
from typing import reveal_type

def get_first(lst: list[int], default: int) -> int:
    if lst:
        return lst[0]
    return default

a = get_first([1, 2, 3], 0)
reveal_type(a)
b = a + 1
reveal_type(b)

 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpi9avy7c5.py:10:12-15: revealed type: int [reveal-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpi9avy7c5.py:12:12-15: revealed type: int [reveal-type]

 INFO 0 errors



In [39]:
%%typecheck
from typing import reveal_type

def get_first(lst: list[str], default: str) -> str:
    if lst:
        return lst[0]
    return default

c = get_first(["a", "b", "c"], "d")
reveal_type(c)
d = c + "e"
reveal_type(d)

 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpti7p_xl6.py:10:12-15: revealed type: str [reveal-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpti7p_xl6.py:12:12-15: revealed type: str [reveal-type]

 INFO 0 errors



In [40]:
%%typecheck
from typing import reveal_type

def get_first(lst: list[int | str], default: int | str) -> int | str:
    if lst:
        return lst[0]
    return default

a = get_first([1, 2, 3], 0)
reveal_type(a)
b = a + 1
reveal_type(b)
c = get_first(["a", "b", "c"], "d")
reveal_type(c)
d = c + "e"
reveal_type(d)

 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmp_jyy_2y3.py:10:12-15: revealed type: int | str [reveal-type]
ERROR /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmp_jyy_2y3.py:11:5-10: `+` is not supported between `str` and `Literal[1]` [unsupported-operation]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmp_jyy_2y3.py:12:12-15: revealed type: int | Unknown [reveal-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmp_jyy_2y3.py:14:12-15: revealed type: int | str [reveal-type]
ERROR /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmp_jyy_2y3.py:15:5-12: `+` is not supported between `int` and `Literal['e']` [unsupported-operation]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmp_jyy_2y3.py:16:12-15: revealed type: int | str [reveal-type]

 INFO 2 errors



### TypeVar, примеры

In [41]:
%%typecheck
# новый синтаксис с 3.12
from typing import reveal_type

def get_first[T: (int, str)](lst: list[T], default: T) -> T:
    if lst:
        return lst[0]
    return default

a = get_first([1, 2, 3], 0)
reveal_type(a)
b = a + 1
reveal_type(b)
c = get_first(["a", "b", "c"], "d")
reveal_type(c)
d = c + "e"
reveal_type(d)

 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpxgxy0o8q.py:11:12-15: revealed type: int [reveal-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpxgxy0o8q.py:13:12-15: revealed type: int [reveal-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpxgxy0o8q.py:15:12-15: revealed type: str [reveal-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpxgxy0o8q.py:17:12-15: revealed type: str [reveal-type]

 INFO 0 errors



In [42]:
%%typecheck
# старый синтаксис (но все еще работает)
import typing as tp
from typing import reveal_type

T = tp.TypeVar('T', int, str)

def get_first(lst: list[T], default: T) -> T:
    if lst:
        return lst[0]
    return default

a = get_first([1, 2, 3], 0)
reveal_type(a)
b = a + 1
reveal_type(b)
c = get_first(["a", "b", "c"], "d")
reveal_type(c)
d = c + "e"
reveal_type(d)

 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpv_zqive8.py:14:12-15: revealed type: int [reveal-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpv_zqive8.py:16:12-15: revealed type: int [reveal-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpv_zqive8.py:18:12-15: revealed type: str [reveal-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpv_zqive8.py:20:12-15: revealed type: str [reveal-type]

 INFO 0 errors



### TypeVar, все же анализатор имеет значение!

In [43]:
%%typecheck
# новый синтаксис с 3.12
from typing import reveal_type

# https://mypy-play.net/?mypy=latest&python=3.13&gist=3d6efd4b89d090c9db78aa5f912c7760 - how it should have been :)

def foo[T: (int, str)](a: T, b: T) -> T:
    reveal_type(a)
    return a + b

a = foo(1, 2)
reveal_type(a)

b = foo("1", "2")
reveal_type(b)

c = foo("1", 2)
reveal_type(c)

 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpbytvoet3.py:8:16-19: revealed type: T [reveal-type]
ERROR /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpbytvoet3.py:9:12-17: `+` is not supported between `T` and `T` [unsupported-operation]
ERROR /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpbytvoet3.py:9:12-17: `+` is not supported between `T` and `T` [unsupported-operation]
ERROR /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpbytvoet3.py:9:12-17: `+` is not supported between `T` and `T` [unsupported-operation]
ERROR /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpbytvoet3.py:9:12-17: Returned type `int | Unknown` is not assignable to declared return type `T` [bad-return]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpbytvoet3.py:12:12-15: revealed type: int [reveal-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpbytvoet3.py:15:12-15: revealed type: str [reveal-type]
ERROR /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpbytvoe

### Overload - не то, что вы подумали
[дока](https://peps.python.org/pep-0484/#function-method-overloading)

In [44]:
%%typecheck
from typing import reveal_type

lst = [1, 2, 3]
reveal_type(lst[0])
reveal_type(lst[0:1])

 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmp2al7nf9u.py:5:12-20: revealed type: int [reveal-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmp2al7nf9u.py:6:12-22: revealed type: list[int] [reveal-type]

 INFO 0 errors



In [45]:
%%typecheck

import typing as tp
from typing import reveal_type

# how it should be: https://mypy-play.net/?mypy=latest&python=3.13&gist=e5a4048236b9e18cab7645148088d03c

class MyIntList:
    def __init__(self, lst: list[int]) -> None:
        self.lst = lst

    @tp.overload
    def __getitem__(self, idx: slice) -> list[int]: ...  # ellipsis is a hack for pyrefly only

    @tp.overload
    def __getitem__(self, idx: int) -> int: ...
    
    def __getitem__(self, idx: slice | int) -> list[int] | int:
        # and this fasle positive is pathetic, meh, mypy is slower, but handles such cases ok
        return self.lst[idx]  # type: ignore[index]

my_lst = MyIntList([1, 2, 3])
reveal_type(my_lst[0])
reveal_type(my_lst[0:1])

 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpunvseydw.py:23:12-23: revealed type: int [reveal-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpunvseydw.py:24:12-25: revealed type: list[int] [reveal-type]

 INFO 0 errors (1 ignored)



### Функциональный тип Callable

In [46]:
%%typecheck

import collections.abc as abc

def g(a: int, b: float) -> float:
    return 1.1

a: abc.Callable[[int, float], float] = g

 INFO 0 errors



### Callable, аннотация типов функции совпадает с тайпингом

In [47]:
%%typecheck
# Привычная функция

import collections.abc as abc

def f(a: abc.Callable[[int, float], float]) -> None:
    pass

def g(a: int, b: float) -> float:
    return 1.1

f(g)

 INFO 0 errors



In [48]:
%%typecheck
# Функция без аргументов

import collections.abc as abc

def f(a: abc.Callable[[], float]) -> None:
    pass

def g() -> float:
    return 1.1

f(g)

 INFO 0 errors



### Callable, часть типов отсутствует

In [49]:
%%typecheck
# Любые аргументы

import collections.abc as abc

def f(a: abc.Callable[..., float]) -> None:
    pass

def g(a: int, b: str, c: int) -> float:
    return 1.1

f(g)

 INFO 0 errors



In [50]:
%%typecheck
# И лямбда тоже

import typing as tp

def f(a: tp.Callable[..., float]) -> None:
    pass

f(lambda a, b, c: 1.1)

 INFO 0 errors



### Callable, передаем функции с другими типами аргументов

In [51]:
%%typecheck
# Повышаем тип аргумента

import collections.abc as abc

def f(a: abc.Callable[[int], float]) -> None:
    pass

def g(a: float) -> float:
    return a

f(g)

 INFO 0 errors



In [52]:
%%typecheck
# Понижаем тип аргумента

import collections.abc as abc

def f(a: abc.Callable[[float], float]) -> None:
    pass

def g(a: int) -> float:
    return a

f(g)

ERROR /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmp0hmn5g_1.py:12:3-4: Argument `(a: int) -> float` is not assignable to parameter `a` with type `(float) -> float` in function `f` [bad-argument-type]

 INFO 1 error



### Callable, передаем функции с другим типом возвращаемого значения

In [53]:
%%typecheck
# Повышаем тип возвращаемого значения

import collections.abc as abc

def f(a: abc.Callable[[int], int]) -> None:
    pass

def g(a: int) -> float:
    return a

f(g)

ERROR /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpfzky2vv9.py:12:3-4: Argument `(a: int) -> float` is not assignable to parameter `a` with type `(int) -> int` in function `f` [bad-argument-type]

 INFO 1 error



In [54]:
%%typecheck
# Понижаем тип возвращаемого значения

import collections.abc as abc

def f(a: abc.Callable[[int], float]) -> None:
    pass

def g(a: int) -> int:
    return a

f(g)

 INFO 0 errors



### Какая вариантность Callable по аргументам? По возвращаемому значению?

### Создание собственных Generic классов

In [55]:
import typing as tp
print(tp.Generic.__doc__)

Abstract base class for generic types.

On Python 3.12 and newer, generic classes implicitly inherit from
Generic when they declare a parameter list after the class's name::

    class Mapping[KT, VT]:
        def __getitem__(self, key: KT) -> VT:
            ...
        # Etc.

On older versions of Python, however, generic classes have to
explicitly inherit from Generic.

After a class has been declared to be generic, it can then be used as
follows::

    def lookup_name[KT, VT](mapping: Mapping[KT, VT], key: KT, default: VT) -> VT:
        try:
            return mapping[key]
        except KeyError:
            return default



### Generic, не так прост как кажется

In [56]:
%%typecheck
import typing as tp
from typing import reveal_type

# Используем тайпвар, как будто пишем дженерик функцию - не работает

T = tp.TypeVar("T", str, int)

class A:
    def __init__(self, a: T) -> None:
        self._a = a
        reveal_type(self._a)
        
    def am(self) -> T:
        reveal_type(self._a)
        return self._a + self._a


a = A(1)
reveal_type(a)
b = a.am()
reveal_type(b)

ERROR /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpf14f1v9n.py:11:14-16: Attribute `_a` cannot depend on type variable `T`, which is not in the scope of class `A` [invalid-type-var]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpf14f1v9n.py:12:20-29: revealed type: T [reveal-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpf14f1v9n.py:15:20-29: revealed type: Unknown [reveal-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpf14f1v9n.py:20:12-15: revealed type: A [reveal-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpf14f1v9n.py:22:12-15: revealed type: @_ [reveal-type]

 INFO 1 error



In [57]:
%%typecheck
import typing as tp
from typing import reveal_type

# корректно - наследоваться от tp.Generic[T] или использовать новый ситаксис!

T = tp.TypeVar("T", int, str)

class A(tp.Generic[T]):    
    def __init__(self, a: T) -> None:
        self._a: T = a
        reveal_type(self._a)
        
    def am(self) -> T:
        reveal_type(self._a)
        return self._a


a = A(1)
reveal_type(a)
b = a.am()
reveal_type(b)

c = A("hello")
reveal_type(c)
d = c.am()
reveal_type(d)

 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpupwinwm4.py:12:20-29: revealed type: T [reveal-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpupwinwm4.py:15:20-29: revealed type: T [reveal-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpupwinwm4.py:20:12-15: revealed type: A[int] [reveal-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpupwinwm4.py:22:12-15: revealed type: int [reveal-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpupwinwm4.py:25:12-15: revealed type: A[str] [reveal-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpupwinwm4.py:27:12-15: revealed type: str [reveal-type]

 INFO 0 errors



In [14]:
%%typecheck
from typing import reveal_type

# новый синтаксис с 3.12

class A[T]:
    def __init__(self, a: T) -> None:
        self._a: T = a
        reveal_type(self._a)
        
    def am(self) -> T:
        reveal_type(self._a)
        return self._a


a = A(1)
reveal_type(a)
b = a.am()
reveal_type(b)

c = A("hello")
reveal_type(c)
d = c.am()
reveal_type(d)

 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpbfxb0wsq.py:9:20-29: revealed type: T [reveal-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpbfxb0wsq.py:12:20-29: revealed type: T [reveal-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpbfxb0wsq.py:17:12-15: revealed type: A[int] [reveal-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpbfxb0wsq.py:19:12-15: revealed type: int [reveal-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpbfxb0wsq.py:22:12-15: revealed type: A[str] [reveal-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpbfxb0wsq.py:24:12-15: revealed type: str [reveal-type]

 INFO 0 errors



In [58]:
%%typecheck
# задаем конкретный тип
from typing import reveal_type

class A[T]:
    def __init__(self) -> None:
        self._a: list[T] = []
        
    def add(self, a: T) -> None:
        self._a.append(a)


a: A[int] = A()
a.add(1)
reveal_type(a)

b: A[float] = A()
b.add("sss")

 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpntpg7qnh.py:15:12-15: revealed type: A[int] [reveal-type]
ERROR /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpntpg7qnh.py:18:7-12: Argument `Literal['sss']` is not assignable to parameter `a` with type `float` in function `A.add` [bad-argument-type]

 INFO 1 error



### Огрангичение тайпваров в новом синтаксисе

In [59]:
%%typecheck
from typing import reveal_type

class A:
    def am(self)-> None:
        pass

class B(A):
    def bm(self) -> None:
        pass

class C[T: A]:
    def __init__(self, obj: T):
        self.obj = obj

    def get_obj(self) -> T:
        return self.obj

a = C(A())
reveal_type(a.get_obj())
a.get_obj().bm()

b = C(B())
reveal_type(b.get_obj())
b.get_obj().bm()



 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpkspxlnhv.py:20:12-25: revealed type: A [reveal-type]
ERROR /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpkspxlnhv.py:21:1-15: Object of class `A` has no attribute `bm` [missing-attribute]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpkspxlnhv.py:24:12-25: revealed type: B [reveal-type]

 INFO 1 error



In [60]:
%%typecheck
from typing import reveal_type

class A:
    def am(self)-> None:
        pass

class B:
    def bm(self) -> None:
        pass

class C[T: (A, B)]:
    def __init__(self, obj: T):
        self.obj = obj

    def get_obj(self) -> T:
        return self.obj

a = C(A())
reveal_type(a.get_obj())
a.get_obj().bm()

b = C(B())
reveal_type(b.get_obj())
b.get_obj().bm()


 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpfwbwjmb_.py:20:12-25: revealed type: A [reveal-type]
ERROR /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpfwbwjmb_.py:21:1-15: Object of class `A` has no attribute `bm` [missing-attribute]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpfwbwjmb_.py:24:12-25: revealed type: B [reveal-type]

 INFO 1 error



###  Переменные классов, Type

In [61]:
# https://www.python.org/dev/peps/pep-0484/#the-type-of-class-objects

import typing as tp
print(tp.Type.__doc__)

Deprecated alias to builtins.type.

    builtins.type or typing.Type can be used to annotate class objects.
    For example, suppose we have the following classes::

        class User: ...  # Abstract base for User classes
        class BasicUser(User): ...
        class ProUser(User): ...
        class TeamUser(User): ...

    And a function that takes a class argument that's a subclass of
    User and returns an instance of the corresponding class::

        def new_user[U](user_class: Type[U]) -> U:
            user = user_class()
            # (Here we could write the user object to a database)
            return user

        joe = new_user(BasicUser)

    At this point the type checker knows that joe has type BasicUser.
    


In [62]:
%%typecheck

a: type[int] = int

class B:
    pass

b: type[B] = B

 INFO 0 errors



###  Type, пример

In [63]:
%%typecheck

class A:
    pass

class B(A):
    pass

def foo(a: type[A]) -> None:
    pass


foo(A())
foo(A)
foo(B())
foo(B)

ERROR /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmps7k3__ag.py:13:5-8: Argument `A` is not assignable to parameter `a` with type `type[A]` in function `foo` [bad-argument-type]
ERROR /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmps7k3__ag.py:15:5-8: Argument `B` is not assignable to parameter `a` with type `type[A]` in function `foo` [bad-argument-type]

 INFO 2 errors



In [64]:
%%typecheck
from typing import reveal_type

class A:
    def __init__(self, a: int) -> None:
        self.a = a

    @classmethod
    def build(cls: type[A]) -> A:
        return cls(1)

class B(A):
    pass
    

reveal_type(A.build())
reveal_type(B.build())

 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpbwxc9g1d.py:16:12-23: revealed type: A [reveal-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpbwxc9g1d.py:17:12-23: revealed type: A [reveal-type]

 INFO 0 errors



In [65]:
%%typecheck
import typing as tp
from typing import reveal_type

# from python 3.11

class A:
    def __init__(self, a: int) -> None:
        self.a = a

    @classmethod
    def build(cls: type[tp.Self]) -> tp.Self:
        return cls(1)

class B(A):
    pass
    

reveal_type(A.build())
reveal_type(B.build())

 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpgiuv1up1.py:19:12-23: revealed type: A [reveal-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpgiuv1up1.py:20:12-23: revealed type: B [reveal-type]

 INFO 0 errors



### Ограничение TypeVar без необходимости перечислять все варианты, ограничение сверху

In [66]:
%%typecheck
import typing as tp
from typing import reveal_type

T = tp.TypeVar('T', bound="A")

class A:
    def __init__(self, a: int) -> None:
        self.a = a

    @classmethod
    def build(cls: type[T]) -> T:
        return cls(1)

class B(A):
    pass
    

reveal_type(A.build())
reveal_type(B.build())

 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmp8x_ki22u.py:19:12-23: revealed type: A [reveal-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmp8x_ki22u.py:20:12-23: revealed type: B [reveal-type]

 INFO 0 errors



In [67]:
%%typecheck
import typing as tp
from typing import reveal_type

class A:
    def __init__(self, a: int) -> None:
        self.a = a

    @classmethod
    def build[T: "A"](cls: type[T]) -> T:
        return cls(1)

class B(A):
    pass
    

reveal_type(A.build())
reveal_type(B.build())

 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpylxa0knr.py:17:12-23: revealed type: A [reveal-type]
 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpylxa0knr.py:18:12-23: revealed type: B [reveal-type]

 INFO 0 errors



###  Какая вариантность у Type?

### Типы → Nominal subtyping, direct inheritance

### Типы → Structural subtyping

### Structural subtyping, примеры

In [68]:
%%typecheck
import collections.abc as abc
import typing as tp
# проставьте тайпинг, чтобы упало только в последнем вызове

def validate_size(a, n) -> None:
    if (a_len := len(a)) > n:
        raise ValueError(
            f"Structure length {a_len} is greater then expected {n}"
        )

        
validate_size([10, 11], 1)  # OK
validate_size(1010, 1)  # fail

 INFO 0 errors



In [69]:
%%typecheck
import collections.abc as abc
import typing as tp
# проставьте тайпинг, чтобы упало только в последнем вызове

def validate_size(a, n) -> None:
    if (a_len := len(a)) > n:
        raise ValueError(
            f"Structure length {a_len} is greater then expected {n}"
        )

validate_size([10, 11], 1)  # OK
validate_size((1, 3, 10), 1)  # OK
validate_size(1010, 1)  # fail


 INFO 0 errors



In [70]:
%%typecheck
import collections.abc as abc
import typing as tp
# проставьте тайпинг, чтобы упало только в последнем вызове

def validate_size(a, n) -> None:
    if (a_len := len(a)) > n:
        raise ValueError(
            f"Structure length {a_len} is greater then expected {n}"
        )


validate_size([10, 11], 1)  # OK
validate_size((1, 3, 10), 1)  # OK
validate_size({1, 3, 10}, 1)  # OK
validate_size(1010, 1)  # fail


 INFO 0 errors



In [71]:
%%typecheck
import collections.abc as abc
import typing as tp
# проставьте тайпинг, чтобы упало только в последнем вызове

class A:
    def __len__(self):
        return 1

def validate_size(a, n) -> None:
    if (a_len := len(a)) > n:
        raise ValueError(
            f"Structure length {a_len} is greater then expected {n}"
        )
        

validate_size([10, 11], 1)  # OK
validate_size((1, 3, 10), 1)  # OK
validate_size({1, 3, 10}, 1)  # OK
validate_size(A(), 1)  # OK
validate_size(1010, 1)  # fail


 INFO 0 errors



### Собственный Structural Subtping - Protocol

In [72]:
%%typecheck

import typing as tp
from typing import reveal_type


class Closeable(tp.Protocol):
    def close(self) -> None:
        pass
    

class A:
    def close(self) -> None:
        print("Close enough")

class B:
    pass


c = open("my_file")


def foo(a: Closeable) -> None:
    reveal_type(a)
    a.close()


foo(A())
foo(B())
foo(c)

 INFO /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmp2bn7fgqe.py:24:16-19: revealed type: Closeable [reveal-type]
ERROR /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmp2bn7fgqe.py:29:5-8: Argument `B` is not assignable to parameter `a` with type `Closeable` in function `foo` [bad-argument-type]

 INFO 1 error



### Protocol and isinstance, runtime_checkable

In [73]:
import typing as tp

# Note that instance checks are not 100% reliable statically, this is why this behavior is opt-in, see section on rejected ideas for examples.
# @see https://peps.python.org/pep-0544/#runtime-checkable-decorator-and-narrowing-types-by-isinstance

@tp.runtime_checkable
class Closeable(tp.Protocol):
    def close(self) -> None:
        pass


class A:
    def close(self):
        print("Close enough")

class B:
    def close(self, message: str) -> None:
        print(message)

isinstance(A(), Closeable), isinstance(B(), Closeable)

(True, True)

### Пример готового Generic + Protocol, просто чтобы вас запутать

In [74]:
%%typecheck

import typing as tp
import collections.abc as abc

def foo(s: abc.Iterable[int]) -> int | None:
    return next(iter(s), None)

class A:
    pass

class B[T]:
    def __iter__(self) -> abc.Iterator[T]:
        return iter([])
    


foo(A())

b: B[int] = B()
foo(b)

c: B[str] = B()
foo(c)

foo([])

ERROR /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpckv7yhk0.py:18:5-8: Argument `A` is not assignable to parameter `s` with type `Iterable[int]` in function `foo` [bad-argument-type]
ERROR /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmpckv7yhk0.py:24:5-6: Argument `B[str]` is not assignable to parameter `s` with type `Iterable[int]` in function `foo` [bad-argument-type]

 INFO 2 errors



### Еще разок про числовые типы

В целом, в питончике творится черная магия с числовыми типами. Есть специальная [иерархия абстрактных типов](https://peps.python.org/pep-3141/) для чисел. А обычные числовые типы являются не прямыми наследниками друг драга, а скорее [структурными](https://mypy.readthedocs.io/en/latest/duck_type_compatibility.html).

In [75]:
from numbers import Real, Integral

isinstance(1, Real), isinstance(1, Integral), isinstance(1.1, Integral)

(True, True, False)

In [76]:
issubclass(float, Real), issubclass(int, float), issubclass(float, Integral)

(True, False, False)

Если кому-то интересно что за чертовщина происходит в ячейке ниже, велкам [в доку](https://peps.python.org/pep-0544/#existing-approaches-to-structural-subtyping).

In [77]:
%%typecheck

# НЕ РАБОТАЕТ для статической проверке типов, только в рантайме. float -> int это чит, которые засунут глубоко в язык

from abc import ABC, abstractmethod

class MyAbstract(ABC):
    @abstractmethod
    def haha(self) -> str:
        pass

class A:
    pass

MyAbstract.register(A)  # does not work for static type checking! Only for runtime

isinstance(A(), MyAbstract)  # True in runtime
issubclass(A, MyAbstract)  # True in runtime


def func(a: MyAbstract) -> None:
    pass

func(A())


ERROR /var/folders/34/ct5sk0352fb4rqp9_r9vn4y00000gn/T/tmp1hyaexo1.py:24:6-9: Argument `A` is not assignable to parameter `a` with type `MyAbstract` in function `func` [bad-argument-type]

 INFO 1 error



# Поздравляю, мы дожили!